# 08. Hands-on: Laya — a multilingual System 1 decision engine

[Laya](https://github.com/NandhaKishorM/laya) is an open-source (Apache-2.0) decision-model family released by Convai Innovations.

- **Non-autoregressive**: BERT-class encoders (English 421M / multilingual 322M) answer choice/noul/score in a single forward pass — the smallest decision models in this workshop (practical even on CPU).
- **100+ languages**: `Router()` detects the input language and automatically routes non-English input (such as Korean) to the `laya-multilingual` checkpoint.
- Returns **calibrated probabilities**, trained with RLCD (RL against strictly proper scoring rules).
- `laya[serve]` also speaks **`/v1/systemone`** (the Jev-compatible wire format), with LangChain/LangGraph/LlamaIndex/CrewAI/MCP integrations and a TypeScript port (`npm install laya-ts`).
- Fully free self-hosting — no API key, no account.

This notebook uses the **native Python `Router` API directly inside the kernel**, with no HTTP server (the kernel env needs `pip install laya` — preinstalled in the workshop kernel; checkpoints download into the HF_HOME cache).


In [ ]:
import sys, os, time
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
from pathlib import Path; sys.path.insert(0, str(Path.cwd().parent))  # ws_common path
os.environ.setdefault("HF_HOME", os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface")))
import ws_common as W
from laya import Router

router = Router()    # downloads a checkpoint on first use
print("laya Router ready")


## 1. Three decision types from one customer message

Laya lets you bundle several questions over one state. In a banking-support context we ask **routing (choice) + urgency (noul) + customer emotion (score)** at once over "My wallet was stolen! Please freeze my card right now!!". With English input, check that `routing.model` reports `english` — non-English input (for example the Korean utterances in the [KR] notebook) is auto-routed to the `laya-multilingual` checkpoint.


In [ ]:
state = "My wallet was stolen! Please freeze my card right now!!"
questions = {
    "d2_team": {"type": "choice",
                "instructions": "Which banking support specialist team should handle this customer message?",
                "criteria": {c: None for c in
                             ["cards", "card_security", "disputes_fees", "transfers",
                              "accounts_balance", "top_up", "exchange_fx", "refunds",
                              "declined_pay", "identity_verif", "account_mgmt", "general_faq"]}},
    "urgent": {"type": "noul", "instructions": "Does this message convey urgency?"},
    "emotion": {"type": "score", "instructions": "How upset is the customer?",
                "criteria": ["calm", "worried", "very upset"]},
}

router.predict(state, questions)   # warm-up (excluded from timing)
t0 = time.perf_counter()
r = router.predict(state, questions)
ms = (time.perf_counter() - t0) * 1000

print(f"checkpoint: {r['routing']['model']}   (3 questions combined: {ms:,.1f}ms)")
a = r["answers"]
print(f"  d2_team -> {a['d2_team']['choice']} (confidence {a['d2_team']['confidence']:.3f})")
print(f"  urgent  -> P(yes) = {a['urgent']['noul']:.3f}")
print(f"  emotion -> score {a['emotion']['score']:.2f}  (0=calm ~ 2=very upset)")

## 2. Plugging it into our LangGraph workflow — a 20-line adapter

Laya is a Python object API, so a **thin adapter** matching the `.decide()` signature drops it straight into the same graph — a demonstration of how small a backend swap is.
> The adapter **pins `model="multilingual"`**. If you leave routing automatic, long states mixed with English (tool JSON etc.) go to the english checkpoint and the repeated checkpoint switching slows everything down; long inputs also need `max_len=8192`. Laya also warns that **choice confidences with 11+ options are uncalibrated** (our D2 has 12) — use the label, avoid confidence-based designs there.


In [ ]:
from agent.decisions import DecisionResult, _fallback

class LayaDecision:
    name = "laya"
    def __init__(self, router):
        self.router = router
    def decide(self, *, state_text, question, choices, decision_id):
        qs = {decision_id: {"type": "choice", "instructions": question,
                            "criteria": {c: None for c in choices}}}
        t0 = time.perf_counter()
        r = self.router.predict(state_text, qs, model="multilingual", max_len=8192)
        ms = (time.perf_counter() - t0) * 1000
        ans = r["answers"][decision_id]
        label, valid = ans["choice"], ans["choice"] in choices
        if not valid:
            label = _fallback(decision_id, choices)
        return DecisionResult(label=label, confidence=ans.get("confidence"),
                              latency_ms=round(ms, 2), valid=valid,
                              usage={}, cost_usd=0.0, raw=str(ans)[:200])

laya_dec = LayaDecision(router)
print("adapter ready")

In [ ]:
from agent.graph_v2 import build_graph_v2
from agent.decisions import SGLangDecision

client, llm = W.make_clients()
graph_laya   = build_graph_v2(laya_dec, llm)
graph_sglang = build_graph_v2(SGLangDecision(port=8001), llm)
sc = W.scenario(1)
print("Scenario:", sc["title"], "|", sc["question"])

In [ ]:
res_laya = W.run_verbose(graph_laya, sc["question"], sc["script"], "Laya")

In [ ]:
res_sglang = W.run_verbose(graph_sglang, sc["question"], sc["script"], "SGLang")

In [ ]:
W.compare(res_laya, res_sglang)

## Measured decision quality on the Korean bench (3-backend comparison)

D1/D2 measured over the workshop's 160-conversation Korean bench (the [KR] track inputs) with objective gold labels (the source project's decider-quality bench, the bench summary (source project)):

| backend | D1 guardrail (3-way) | D2 routing strict/lenient (12-way) | p50 latency |
|---|---|---|---|
| SGLang Qwen3-VL-8B | **0.913** | **0.629 / 0.707** | 33ms |
| strands-decider 2B | 0.613 | 0.500 / 0.593 | 85ms |
| Laya multilingual 322M | 0.419 | 0.357 / 0.379 | 21ms |

**Reading**: model capacity dominated the quality ceiling (8B > 2B > 0.3B), beating language specialization (Laya multilingual). The Laya number matches its own documented zero-shot score (0.362); Laya is a **"fine-tune me" model** — RLCD fine-tuning lifts it to 0.766. Conversely, smaller means better latency and footprint — a clean three-way quality/cost/speed trade-off.


## 3. Wrap-up — a map of the decision-model options

Including this folder (40_other_deciders), you have now experienced the same banking-support decisions five different ways.

| Approach | Decision maker | Decision latency (this env) | Cost structure | Notes |
|---|---|---|---|---|
| Bedrock Haiku 4.5 | General LLM generation | ~640ms | Token billing | Quality baseline, managed |
| SGLang /v1/systemone (01-05) | The Qwen3-VL-8B you already serve | **~35ms** | ~zero marginal cost | Multimodal, reuses existing serving |
| Bedrock Intelligent Routing (06) | Picks the **generation model** (not a decision) | Selected model's generation time | $1/1,000 req + tokens | Generation cost optimizer |
| strands-decider 2B (07) | Dedicated decision model | ~80ms | Self-hosted | Calibrated confidence, Strands SDK integration |
| Laya 322-421M (08) | Dedicated decision model | **~30ms** (in-process) | Self-hosted (CPU OK) | Multilingual auto-routing, smallest footprint |

- **The Korean angle**: Laya auto-selects its multilingual checkpoint and treats Korean as a first-class citizen — the dedicated model that fits this workshop's Korean scenarios most naturally.
- **Caution**: the meaning of "confidence" (how it is calibrated) differs per vendor. Thresholds do not transfer between products — always re-measure reliability on your own data.
- When zero-shot is not enough, Laya ships an RLCD fine-tuning path (official notebooks) — base 0.362 -> fine-tuned 0.766 on the typed-decisions benchmark.
